# 🔬 Native Python: Under The Hood & Complete Language Internals

A definitive masterclass on pure, native Python (CPython). This notebook covers everything built into the runtime engine: keywords, built-in functions, constants, object model, operators, decorators, magic (dunder) methods, standard library architecture, memory management, garbage collection, and bytecode execution.

---

## 📑 Table of Contents
1. [Keywords & Soft Keywords](#1.-Keywords-&-Soft-Keywords)
2. [Built-in Constants & Sentinels](#2.-Built-in-Constants-&-Sentinels)
3. [Complete Built-in Functions Encyclopedia (All 71 Built-ins)](#3.-Complete-Built-in-Functions-Encyclopedia)
4. [Built-in Type Hierarchy & The Object Model (`type` vs `object`)](#4.-Built-in-Type-Hierarchy-&-The-Object-Model)
5. [Operators, Precedence & The `operator` Module](#5.-Operators,-Precedence-&-The-operator-Module)
6. [Decorators: Mechanics & Built-in Decorators](#6.-Decorators:-Mechanics-&-Built-in-Decorators)
7. [Magic Methods (Dunder Methods) Encyclopedia](#7.-Magic-Methods-(Dunder-Methods)-Encyclopedia)
8. [Standard Library Architecture & Key Native Modules](#8.-Standard-Library-Architecture-&-Key-Native-Modules)
9. [Memory Management & Garbage Collection (Ref Counting + Cyclical GC)](#9.-Memory-Management-&-Garbage-Collection)
10. [Under The Hood: Compilation, Bytecode & The Python Virtual Machine](#10.-Under-The-Hood:-Compilation,-Bytecode-&-The-Python-Virtual-Machine)

---

## 1. Keywords & Soft Keywords

Python reserves specific identifiers as **keywords** that define language grammar and control flow. Keywords cannot be used as regular variable or function names.

- **Hard Keywords** (`keyword.kwlist`): Strictly reserved in all contexts.
- **Soft Keywords** (`keyword.softkwlist`): Reserved only in specific syntactic positions (e.g., `match`, `case`, `_`, `type`).

In [1]:
import keyword

print(f"Total Hard Keywords: {len(keyword.kwlist)}")
print("Hard Keywords:")
for i in range(0, len(keyword.kwlist), 5):
    print("  ".join(f"{kw:<12}" for kw in keyword.kwlist[i:i+5]))

print(f"\nTotal Soft Keywords: {len(keyword.softkwlist)}")
print("Soft Keywords:", keyword.softkwlist)

# Testing keyword validation
print("\niskeyword('async'):", keyword.iskeyword('async'))
print("issoftkeyword('match'):", keyword.issoftkeyword('match'))

Total Hard Keywords: 35
Hard Keywords:
False         None          True          and           as          
assert        async         await         break         class       
continue      def           del           elif          else        
except        finally       for           from          global      
if            import        in            is            lambda      
nonlocal      not           or            pass          raise       
return        try           while         with          yield       

Total Soft Keywords: 4
Soft Keywords: ['_', 'case', 'match', 'type']

iskeyword('async'): True
issoftkeyword('match'): True


## 2. Built-in Constants & Sentinels

Python exposes several core singleton constants living in the built-in namespace:

| Constant | Type | Description |
| :--- | :--- | :--- |
| `True` / `False` | `bool` | Boolean truth values (subclass of `int`, `int(True) == 1`). |
| `None` | `NoneType` | Universal sentinel denoting absence of a value. Singleton in memory. |
| `Ellipsis` / `...` | `ellipsis` | Extended slice syntax, type annotations, and placeholder sentinel. |
| `NotImplemented` | `NotImplementedType` | Special value returned by binary dunder methods when an operation is unsupported. |
| `__debug__` | `bool` | True unless Python is executed with the `-O` (optimize) flag. |

In [2]:
# Identity and Memory Inspection of Built-in Constants
print("id(None)        :", hex(id(None)))
print("id(...)         :", hex(id(...)))
print("id(Ellipsis)    :", hex(id(Ellipsis)), "(Identical to ... :", ... is Ellipsis, ")")
print("NotImplemented  :", type(NotImplemented))
print("__debug__ active:", __debug__)

# Ellipsis in Type Hints or multi-dimensional slicing:
from typing import Callable
AnyFunc = Callable[..., int]  # Takes any arguments, returns int
print("Ellipsis used in type annotation:", AnyFunc)

id(None)        : 0xa408a0
id(...)         : 0xa42740
id(Ellipsis)    : 0xa42740 (Identical to ... : True )
NotImplemented  : <class 'NotImplementedType'>
__debug__ active: True
Ellipsis used in type annotation: typing.Callable[..., int]


## 3. Complete Built-in Functions Encyclopedia

Python provides 71 native functions directly in `builtins` without needing imports. They are organized below into functional categories:

1. **Mathematical & Numeric**: `abs`, `divmod`, `max`, `min`, `pow`, `round`, `sum`
2. **Type Constructors**: `bool`, `bytearray`, `bytes`, `complex`, `dict`, `float`, `frozenset`, `int`, `list`, `memoryview`, `object`, `set`, `str`, `tuple`, `type`
3. **Iterators & Sequences**: `all`, `any`, `enumerate`, `filter`, `iter`, `len`, `map`, `next`, `range`, `reversed`, `slice`, `sorted`, `zip`, `aiter`, `anext`
4. **Object Introspection & Attribute Access**: `callable`, `delattr`, `dir`, `getattr`, `hasattr`, `hash`, `id`, `isinstance`, `issubclass`, `repr`, `ascii`, `vars`
5. **Scope & Execution Context**: `eval`, `exec`, `compile`, `globals`, `locals`, `breakpoint`
6. **I/O & System Services**: `print`, `input`, `open`, `format`, `help`, `__import__`
7. **Class & Method Decorators**: `classmethod`, `staticmethod`, `property`, `super`

In [3]:
import builtins

all_builtins = [name for name in dir(builtins) if not name.startswith('_') and callable(getattr(builtins, name))]
print(f"Total callable built-in functions/constructors: {len(all_builtins)}\n")

# Quick demonstration of less common but powerful built-ins:
# 1. divmod (returns quotient and remainder simultaneously)
q, r = divmod(29, 5)
print(f"divmod(29, 5) -> quotient={q}, remainder={r}")

# 2. all / any
print("all([True, 1, 'text']):", all([True, 1, 'text']))
print("any([False, 0, '']):", any([False, 0, '']))

# 3. hash & id
s = "Python CPython"
print(f"hash('{s}') -> {hash(s)}")

# 4. compile, eval, exec
code_obj = compile("x * 2 + 10", filename="<string>", mode="eval")
print("eval(compiled_code, {'x': 15}) ->", eval(code_obj, {"x": 15}))

Total callable built-in functions/constructors: 146

divmod(29, 5) -> quotient=5, remainder=4
all([True, 1, 'text']): True
any([False, 0, '']): False
hash('Python CPython') -> 5973311752379999345
eval(compiled_code, {'x': 15}) -> 40


## 4. Built-in Type Hierarchy & The Object Model (`type` vs `object`)

In Python, **everything is an object**, and every object has:
1. An **Identity** (`id(x)` - memory address in CPython).
2. A **Type** (`type(x)` - determines supported operations).
3. A **Value** (the data held).

### The Fundamental Metaclass Relationship:
- `object` is the base class for all classes in Python (`issubclass(X, object)` is True for everything).
- `type` is the default metaclass that creates all classes.
- `isinstance(object, type)` is `True` (object was constructed by type).
- `issubclass(type, object)` is `True` (type inherits from object).

In [4]:
# The bootstrap circle of Python's type system
print("type(object)               :", type(object))          # object is an instance of type
print("type(type)                 :", type(type))            # type is an instance of itself!
print("issubclass(type, object)   :", issubclass(type, object)) # type inherits from object
print("isinstance(object, type)   :", isinstance(object, type))
print("isinstance(type, object)   :", isinstance(type, object))

# Dynamic class creation using type(name, bases, dict)
DynamicRobot = type('DynamicRobot', (), {
    'version': '1.0',
    'speak': lambda self: 'Beep Boop, I am dynamically generated!'
})

bot = DynamicRobot()
print("\nDynamic Class Instance:", bot.speak())
print("Class MRO (Method Resolution Order):", DynamicRobot.__mro__)

type(object)               : <class 'type'>
type(type)                 : <class 'type'>
issubclass(type, object)   : True
isinstance(object, type)   : True
isinstance(type, object)   : True

Dynamic Class Instance: Beep Boop, I am dynamically generated!
Class MRO (Method Resolution Order): (<class '__main__.DynamicRobot'>, <class 'object'>)


## 5. Operators, Precedence & The `operator` Module

Python provides a complete suite of operators:
- **Arithmetic**: `+`, `-`, `*`, `/`, `//`, `%`, `**`, `@` (Matrix Multiplication - PEP 465)
- **Bitwise**: `&` (AND), `|` (OR), `^` (XOR), `~` (NOT), `<<` (Left Shift), `>>` (Right Shift)
- **Assignment & Walrus**: `=`, `+=`, `-=`, etc., and `:=` (Assignment Expression / Walrus Operator - PEP 572)
- **Comparison & Identity**: `==`, `!=`, `<`, `<=`, `>`, `>=`, `is`, `is not`, `in`, `not in`

Every operator maps directly to a corresponding function in the standard library `operator` module.

In [5]:
import operator

# Matrix Multiplication operator '@'
class Matrix2x2:
    def __init__(self, data): self.data = data
    def __matmul__(self, other):
        # Simplified dot product mock for demo
        return Matrix2x2([[sum(a * b for a, b in zip(row, col)) for col in zip(*other.data)] for row in self.data])
    def __repr__(self): return str(self.data)

m1 = Matrix2x2([[1, 2], [3, 4]])
m2 = Matrix2x2([[2, 0], [1, 2]])
print("Matrix @ Multiplication :", m1 @ m2)

# Walrus Operator ':='
sample_line = "antigravity"
if (n := len(sample_line)) > 5:
    print(f"Length captured via walrus (:=): {n}")

# The operator module alternatives
print("operator.add(10, 20)       :", operator.add(10, 20))
print("operator.itemgetter(1)([a,b,c]):", operator.itemgetter(1)(['first', 'target', 'third']))

Matrix @ Multiplication : [[4, 4], [10, 8]]
Length captured via walrus (:=): 11
operator.add(10, 20)       : 30
operator.itemgetter(1)([a,b,c]): target


## 6. Decorators: Mechanics & Built-in Decorators

Decorators in Python are higher-order functions that accept a callable and return an enhanced callable.

### Syntax Sugar Equivalence:
```python
@my_decorator
def func(): pass
# Is identical to: func = my_decorator(func)
```

### Core Built-in & Standard Library Decorators:
- `@property`, `@<prop>.setter`, `@<prop>.deleter`: Encapsulate getters/setters via the Descriptor protocol.
- `@classmethod`: Passes the class `cls` as the first argument.
- `@staticmethod`: Regular function attached to class namespace without `self` or `cls`.
- `@functools.wraps`: Preserves function introspection (`__name__`, `__doc__`, `__annotations__`).
- `@functools.lru_cache`: Memoization caching for pure function results.

In [6]:
import functools, time

# 1. Decorator under the hood using functools.wraps
def timer_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        duration = time.perf_counter() - start
        print(f"[{func.__name__}] executed in {duration * 1e6:.2f} µs")
        return result
    return wrapper

@timer_decorator
def compute_heavy_task(n: int) -> int:
    return sum(i * i for i in range(n))

# 2. Built-in caching decorator @functools.lru_cache
@functools.lru_cache(maxsize=128)
def fibonacci(n: int) -> int:
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print("Heavy task result :", compute_heavy_task(10000))
print("Fibonacci(20)     :", fibonacci(20))
print("Cache info        :", fibonacci.cache_info())

# 3. @property, @classmethod, @staticmethod demo
class Temperature:
    def __init__(self, celsius: float): self._c = celsius
    @property
    def fahrenheit(self) -> float: return (self._c * 9/5) + 32
    @classmethod
    def from_fahrenheit(cls, f: float): return cls((f - 32) * 5/9)
    @staticmethod
    def absolute_zero_c(): return -273.15

t = Temperature(25.0)
print(f"25°C in Fahrenheit: {t.fahrenheit}°F")
print("Absolute Zero C   :", Temperature.absolute_zero_c())


[compute_heavy_task] executed in 330.02 µs
Heavy task result : 333283335000
Fibonacci(20)     : 6765
Cache info        : CacheInfo(hits=18, misses=21, maxsize=128, currsize=21)
25°C in Fahrenheit: 77.0°F
Absolute Zero C   : -273.15


## 7. Magic Methods (Dunder Methods) Encyclopedia

Magic methods (prefixed and suffixed with double underscores `__`) hook user classes directly into Python's C-level evaluation engine.

### Comprehensive Categorization:
1. **Lifecycle & Instantiation**: `__new__` (allocates memory), `__init__` (initializes), `__del__` (destructor).
2. **String Representations**: `__repr__` (unambiguous code representation), `__str__` (human readable), `__bytes__`, `__format__`.
3. **Rich Comparisons**: `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`, `__hash__`.
4. **Attribute Access & Descriptors**: `__getattr__`, `__getattribute__`, `__setattr__`, `__delattr__`, `__dir__`; Descriptor protocol: `__get__`, `__set__`, `__delete__`, `__set_name__`.
5. **Container & Emulating Sequences**: `__len__`, `__getitem__`, `__setitem__`, `__delitem__`, `__iter__`, `__next__`, `__reversed__`, `__contains__`.
6. **Arithmetic & Bitwise**: `__add__`, `__radd__` (reflected), `__iadd__` (in-place), `__sub__`, `__mul__`, `__truediv__`, `__floordiv__`, `__mod__`, `__pow__`, `__matmul__`.
7. **Context Managers**: `__enter__`, `__exit__`, and async variants `__aenter__`, `__aexit__`.
8. **Callables & Meta**: `__call__` (make instance callable like function), `__init_subclass__`, `__class_getitem__`.

In [7]:
# Complete Custom Class exercising multiple Dunder Protocols
class VectorND:
    def __new__(cls, *coords):
        # Allocates instance before __init__
        instance = super().__new__(cls)
        return instance

    def __init__(self, *coords):
        self._coords = tuple(float(c) for c in coords)

    # Representation
    def __repr__(self): return f"VectorND{self._coords}"
    def __str__(self): return f"<{' , '.join(str(c) for c in self._coords)}>"

    # Container Emulation
    def __len__(self): return len(self._coords)
    def __getitem__(self, idx): return self._coords[idx]
    def __iter__(self): return iter(self._coords)
    def __contains__(self, val): return val in self._coords

    # Arithmetic & Reflected
    def __add__(self, other):
        if isinstance(other, VectorND) and len(self) == len(other):
            return VectorND(*(a + b for a, b in zip(self, other)))
        return NotImplemented

    # Equality & Hashing
    def __eq__(self, other):
        return isinstance(other, VectorND) and self._coords == other._coords
    def __hash__(self):
        return hash(self._coords)

    # Callable
    def __call__(self, scalar):
        return VectorND(*(c * scalar for c in self))

v1 = VectorND(1, 2, 3)
v2 = VectorND(4, 5, 6)
v3 = v1 + v2

print("v1 + v2           :", v3)
print("Vector Length     :", len(v3))
print("Index v3[0]       :", v3[0])
print("Scaled (callable) :", v3(2.0))
print("Hashable in Set   :", {v1, v2})

v1 + v2           : <5.0 , 7.0 , 9.0>
Vector Length     : 3
Index v3[0]       : 5.0
Scaled (callable) : <10.0 , 14.0 , 18.0>
Hashable in Set   : {VectorND(1.0, 2.0, 3.0), VectorND(4.0, 5.0, 6.0)}


## 8. Standard Library Architecture & Key Native Modules

Python adheres to the "batteries included" philosophy. The standard library is categorized into core architectural domains:

| Domain | Core Native Modules |
| :--- | :--- |
| **Data Structures & Algorithms** | `collections` (`deque`, `Counter`, `defaultdict`), `heapq`, `bisect`, `itertools`, `functools` |
| **Runtime & System Hooks** | `sys`, `os`, `platform`, `gc`, `types`, `inspect`, `builtins`, `site` |
| **Text & Pattern Matching** | `re`, `string`, `unicodedata`, `difflib`, `textwrap` |
| **Files, Paths & Serialization** | `pathlib`, `shutil`, `tempfile`, `pickle`, `json`, `csv`, `sqlite3` |
| **Math & Number Theory** | `math`, `cmath`, `decimal`, `fractions`, `random`, `statistics` |
| **Concurrency & Async** | `asyncio`, `threading`, `multiprocessing`, `concurrent.futures`, `queue` |
| **Typing & Metaprogramming** | `typing`, `types`, `dataclasses`, `enum`, `abc`, `contextlib` |
| **Bytecode & Compiler Internals** | `dis`, `ast`, `symtable`, `code`, `opcode` |

In [8]:
from collections import Counter, defaultdict, deque
import itertools
from pathlib import Path

# 1. High-performance collections
counts = Counter("abracadabra")
print("Counter top 2     :", counts.most_common(2))

# 2. itertools combinations & permutations
combos = list(itertools.combinations(['A', 'B', 'C'], 2))
print("Combinations of 3 :", combos)

# 3. Modern Pathlib object-oriented pathing
p = Path(".")
print("Resolved Path     :", p.resolve().name)

Counter top 2     : [('a', 5), ('b', 2)]
Combinations of 3 : [('A', 'B'), ('A', 'C'), ('B', 'C')]
Resolved Path     : NOTEBOOKS


## 9. Memory Management & Garbage Collection

CPython combines **Reference Counting** with a **Generational Cyclic Garbage Collector** (`gc`).

### 9.1 Reference Counting (Primary Reclamation Mechanism)
- Every Python object (`PyObject`) in C has an internal `ob_refcnt` field.
- When a reference is created, `ob_refcnt` increments. When deleted, it decrements.
- When `ob_refcnt == 0`, memory is **immediately freed** back to Python's memory pool.
- Inspected via `sys.getrefcount(obj)` (note: passing an object to `getrefcount()` temporarily adds 1 reference).

### 9.2 The Cyclic Garbage Collector (Resolving Cycles)
- Reference counting alone fails when objects form reference cycles (`A -> B -> A`).
- CPython's generational GC tracks container objects (`list`, `dict`, `set`, class instances).
- Divides objects into **3 Generations**: Generation 0 (young/new), Generation 1, and Generation 2 (old/survivors).
- Young generations are collected frequently; surviving objects are promoted to older generations.

### 9.3 The CPython Memory Allocator (PyMalloc)
- Small objects (<= 512 bytes) are managed by `PyMalloc`.
- Organizes memory into **Arenas** (256 KB) -> **Pools** (4 KB) -> **Blocks** (8 to 512 bytes).
- Avoids OS-level `malloc`/`free` overhead and fragmentation.

In [9]:
import sys, gc

# 1. Reference Counting
my_list = [10, 20, 30]
print(f"Initial refcount for my_list: {sys.getrefcount(my_list) - 1}")  # subtract 1 for getrefcount argument

alias = my_list
print(f"Refcount after creating alias: {sys.getrefcount(my_list) - 1}")
del alias
print(f"Refcount after deleting alias: {sys.getrefcount(my_list) - 1}")

# 2. Creating a Cyclic Reference
class Node:
    def __init__(self, name): 
        self.name = name
        self.cycle = None

node_a = Node("NodeA")
node_b = Node("NodeB")
node_a.cycle = node_b
node_b.cycle = node_a  # Cycle created!

del node_a
del node_b
# At this point, refcounts > 0 due to the internal cycle, but they are unreachable!

# Trigger explicit GC collection
collected_unreachable = gc.collect()
print(f"Cyclic garbage collected unreachable objects: {collected_unreachable}")
print("GC thresholds (Gen0, Gen1, Gen2)            :", gc.get_threshold())

Initial refcount for my_list: 1
Refcount after creating alias: 2
Refcount after deleting alias: 1
Cyclic garbage collected unreachable objects: 5
GC thresholds (Gen0, Gen1, Gen2)            : (700, 10, 10)


## 10. Under The Hood: Compilation, Bytecode & The Python Virtual Machine

How Python actually runs your code:

```text
  Source Code (.py) 
         │
         ▼ (Tokenizer & Parser: Abstract Syntax Tree - AST)
     AST Tree
         │
         ▼ (Compiler)
   Bytecode (PyCodeObject)
         │
         ▼
 CPython Virtual Machine (Evaluation Loop - Python/ceval.c)
         │
         ▼
  Native Machine Execution
```

### Key Concepts:
- **Code Objects** (`types.CodeType`): Immutable compiled bytecode with constant tables (`co_consts`), local variable names (`co_varnames`), and raw byte instructions (`co_code`).
- **Stack-Based Evaluation**: Python's VM is a stack machine. Instructions push and pop objects from an evaluation value stack.
- **The Global Interpreter Lock (GIL)**: A mutual-exclusion lock preventing multiple native OS threads from executing Python bytecodes simultaneously, ensuring thread-safe reference count mutations.

In [10]:
import dis

# Target function to inspect
def compute_hypotenuse(a: float, b: float) -> float:
    sum_sq = a**2 + b**2
    return sum_sq ** 0.5

print("=== Code Object Properties ===")
co = compute_hypotenuse.__code__
print("co_varnames (locals) :", co.co_varnames)
print("co_consts (constants):", co.co_consts)
print("co_argcount          :", co.co_argcount)
print("co_stacksize         :", co.co_stacksize)

print("\n=== Bytecode Disassembly via dis.dis() ===")
dis.dis(compute_hypotenuse)

=== Code Object Properties ===
co_varnames (locals) : ('a', 'b', 'sum_sq')
co_consts (constants): (None, 2, 0.5)
co_argcount          : 2
co_stacksize         : 3

=== Bytecode Disassembly via dis.dis() ===
  4           0 RESUME                   0

  5           2 LOAD_FAST                0 (a)
              4 LOAD_CONST               1 (2)
              6 BINARY_OP                8 (**)
             10 LOAD_FAST                1 (b)
             12 LOAD_CONST               1 (2)
             14 BINARY_OP                8 (**)
             18 BINARY_OP                0 (+)
             22 STORE_FAST               2 (sum_sq)

  6          24 LOAD_FAST                2 (sum_sq)
             26 LOAD_CONST               2 (0.5)
             28 BINARY_OP                8 (**)
             32 RETURN_VALUE
